# Ollama 极速部署
Ollama、GGUF 机制与 LangChain 本地 Agent 缝合

1. 解构 GGUF 存储格式：理解 llama.cpp 生态下的 GGUF 单文件模型格式及其 CPU/GPU 混合加载（Offloading）原理。
2. 掌握 Ollama 部署与 CLI 管理：掌握使用 Ollama 一键拉起、管理及通过 Modelfile 自定义模型参数（System Prompt、Temperature、Context Window）
3. 使用 LangChain / LangGraph 绑定 Ollama 本地模型，构建全私有化本地 Agent。

## GGUF 架构格式与 Ollama 运行机制

[ GGUF 单文件结构 ]
┌───────────────────────────────┐
│ Header (Magic, Version, etc.) │
├───────────────────────────────┤
│ Metadata (KV Pairs, Prompt...)│
├───────────────────────────────┤
│ Tensor Info & Quantized Weights│ (Q4_K_M, Q5_K_S, Q8_0...)
└───────────────────────────────┘


#### 什么是 GGUF (GGML Unified Format)？
GGUF 是由开源社区 `llama.cpp` 主导设计的新一代大模型存储文件格式，旨在替代旧版 GGML 格式：
* 单文件开箱即用：将模型的 Tokenizer 词表、架构超参数（Metadata）以及量化后的张量权重（Tensors）全部打包封装在单一的 `.gguf` 文件中，极大降低了部署门槛。
* 极致的 CPU/GPU 混合加载 (Layer Offloading)：GGUF 允许将模型的神经网络层逐层切分，例如将 32 层中的 20 层推送到 GPU 显存，剩余 12 层保留在 CPU 内存（RAM）中运行。这意味着即使显存不足，也能以稍低的速度强行跑起大模型。
* 丰富的 K-quant 量化级别：支持 `Q4_K_M`（4-bit 组合量化，推荐最佳平衡）、`Q5_K_M`、`Q8_0` 等多种精细度控制。

#### Ollama 的包装与调度哲学
Ollama 是基于 `llama.cpp` 构建的高层 Go 语言服务框架。它将复杂的 C++ 编译与 GGUF 模型文件管理包装成了类似于 Docker 的容器化 CLI 体验：
* 使用 `ollama run qwen2.5` 即可自动完成下载、显存/内存切片分配以及后台 RESTful API 服务的启动。

#### 基于 Ollama + LangChain 构建全私有化 Agent
> 环境准备提示：
> 1. 请确保本地已安装并启动 Ollama（官网下载 `ollama.com`），并在终端运行：`ollama pull qwen2.5:7b`
> 2. 安装 Python 依赖：`pip install langchain-community langchain-core`

In [ ]:
from langchain_community.llms import Ollama
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ================= 1. 初始化 ChatOllama 实例 =================
# 指向本地默认运行的 Ollama 服务 (http://localhost:11434)
local_llm = ChatOllama(
    model="qwen2.5:7b",       # 指定 Ollama 本地已下载的模型 Tag
    temperature=0.2,           # 降低随机性
    num_ctx=4096,              # 显式设定上下文窗口大小 (Context Window)
)

print("🚀 正在验证 Ollama 本地模型连接...")

# ================= 2. 测试原生 ChatMessage 交互 =================
messages = [
    SystemMessage(content="你是一名部署在本地私有环境的 AI 助理，不能将任何数据传输到外网。"),
    HumanMessage(content="请简述 Ollama 框架的三大优势。")
]

response = local_llm.invoke(messages)
print(f"\n💬 [Ollama 本地模型回复]:\n{response.content}\n")

# ================= 3. 与 LangChain LCEL 表达链 (Chain) 无缝缝合 =================
print("🔗 正在构建 LangChain 本地 LCEL 链...")

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个专业的 Python 运维专家。请只给出最简洁的代码示例，不要有废话。"),
    ("user", "写一段 Python 代码：检查本地端口 {port} 是否被占用。")
])

# 拼装 LCEL 链：Prompt -> Local LLM -> StrOutputParser
chain = prompt_template | local_llm | StrOutputParser()

# 执行流式输出 (Streaming)
print("\n⚡ [LCEL 流式响应结果]:")
for chunk in chain.stream({"port": 8000}):
    print(chunk, end="", flush=True)
print("\n")


在终端运行 `ollama create my-custom-agent -f Modelfile`，即可在代码中直接指定 `model="my-custom-agent"`。

**思考**
1. Tool Calling 适配：对于某些不支持原生 JSON Function Calling 的小型开源模型（如 3B/7B），在 Ollama 中通过 LangChain 绑定 `.bind_tools()` 时容易报错。作为架构师，你应该如何在 Prompt 层使用 ReAct 或 JSON Schema 格式做手动的格式强制提取？
2. GPU 层级加载 (Layer Offloading) 调优：在 GPU 显存紧张的设备上运行 Ollama 时，命令行环境变量 `OLLAMA_NUM_PARALLEL`（控制并发数）与 `num_gpu`（加载到 GPU 的 Transformer 层数）应该如何权衡以避免 OOM 崩溃？

## 开源大模型部署全栈复盘与私有化 Agent 架构自测
1. 开源大模型全栈部署技术选型：理清原生 `transformers`、`vLLM` 与 `Ollama`/`llama.cpp` 在不同业务场景下的最佳适用界限。
2. 构建全私有化 Agent 系统架构：整合本地高并发 LLM 服务与先前所学的 LangGraph，打通“数据不出域、工具本地调用、高并发响应”的企业级 Agent 终极形态。
3. 通过硬核工程场景考题，检验你对本地模型推理加速、显存预算与量化部署的掌握程度。

#### 核心技术栈全景复盘

| 方案 / 框架                       | 核心底层技术                                  | 显存/硬件要求                            | 优势                                           | 劣势 / 局限                           | 最佳适用场景                               |
|-------------------------------|-----------------------------------------|------------------------------------|----------------------------------------------|-----------------------------------|--------------------------------------|
| Hugging Face (`transformers`) | PyTorch 原生前向传播                          | 较高（需加载全量 FP16/BF16 权重与连续 KV Cache） | 灵活性极高，原生支持模型结构修改、梯度提取与微调。                    | 吞吐量低，无连续批处理，长上下文容易发生显存碎片化（OOM）    | 模型研究、算法开发、离线小批量评测与微调实验。              |
| vLLM 引擎                       | PagedAttention + Continuous Batching    | 中等至高等（需要 NVIDIA GPU / CUDA 环境）     | 极高并发吞吐量、显存利用率近乎 100%、原生兼容 OpenAI API 规范。     | 不支持 CPU 混合加载，对消费级单显卡的小型边缘设备适配较繁琐。 | 生产级 API 服务、高并发 Agent 后端、企业级私有云大模型平台。 |
| AWQ / GPTQ 量化                 | 4-bit 权重量化 (PTQ)                        | 低（7B 模型仅需 4.5~5GB 显存）              | 显存占用立减 70%，在 vLLM 中配合 CUDA 算子可实现极致吞吐。        | 量化过程需要校准集，极少部分特殊格式或数学推演能力可能轻微受损。  | 消费级显卡部署、高性价比边缘节点部署。                  |
| Ollama / `llama.cpp`          | GGUF 格式 + Layer Offloading (CPU/GPU 混合) | 极低（支持 GPU+RAM 联合加载，无专用显卡亦可运行）      | 开箱即用、单文件便携、跨平台（macOS / Linux / Windows）适配完美。 | 在超高并发场景下的总吞吐量（Throughput）不及 vLLM。 | 本地桌面端助手、私有化 Agent 快速原型验证、边缘轻量级服务。    |

#### 企业级全私有化 Agent 架构设计规范
将本地开源大模型推理引擎与 LangGraph 结合时，推荐采用如下的分层架构设计：

[ 前端交互层 / API ]
         ↓ (HTTP / WebSocket)
[ Agent 逻辑编排层 (LangGraph / LangChain) ]
    ├── 状态管理 & 动态路由 (StateGraph)
    ├── 工具调用 & 本地 Agent 逻辑 (Tools / ReAct Loop)
    └── 记忆与 Checkpointer (Local SQLite / Redis)
         ↓ (OpenAI 协议兼容接口: http://localhost:8000/v1)
[ 高吞吐本地推理层 (vLLM API Server / AWQ INT4) ]
         ↓ (PagedAttention / CUDA Kernels)
[ 物理硬件层 (NVIDIA GPU Cluster / 本地工作站) ]

* 安全防线：网络层全局隔离，Agent 与推理引擎运行在同一局域网（VPC）内，完全脱离外部云端 API 依赖。
* 协议统一：Agent 编排层统一采用 OpenAI 客户端协议，使得后续在“云端商业模型（GPT-4o）”与“本地开源模型（Qwen2.5/Llama3）”之间进行无缝无感切换。

**思考**

1. 考核题 1（显存预算与部署选型）：
    * 你的公司有一台配有单张 NVIDIA RTX 4090（24GB 显存）的服务器，需要部署一个 Qwen2.5-14B-Instruct 模型，同时服务内部 10 名员工的并发提问，且要求 Context Window 达到 4096 Tokens。
    * 如果使用原生的 FP16 加载，24GB 显存能否满足需求？若无法满足，你会选择 vLLM + AWQ 4-bit 量化 还是 Ollama + GGUF？请给出具体的显存计算过程与选型理由。

2. 考核题 2（PagedAttention 与并发瓶颈）：
    * 在使用 vLLM 部署 API 服务时，如果发现客户端请求出现较高的 TTFT（Time to First Token，首字延迟），但一旦开始生成，后续 Token 的吐字速度极快（High Throughput）。
    * 请问这通常是由于什么原因造成的？可以通过调节 vLLM 启动命令行中的哪些参数（如 `--max-num-batched-tokens` 或 `--gpu-memory-utilization`）来进行调优？

3. 考核题 3（全私有化 Agent 调优）：
    * 将本地加载的 7B/14B 开源模型（如 Qwen2.5）接入 LangGraph 构建 ReAct 工具调用 Agent 时，有时模型会输出不符合标准的 JSON 格式，导致 Tool Calling 解析失败。
    * 请给出两种在不改变大模型权重的前提下，提升本地 Agent 工具调用成功率与鲁棒性的工程优化手段。